# 🎯 DGNet: Camouflage Object Detection (COD) Pretraining on Kaggle Benchmark
### Self-Contained Google Colab / Kaggle Training Pipeline (Zero Local Imports)

This notebook is **100% self-contained** for running on **Google Colab**, **Kaggle Notebooks**, or any cloud GPU instance. It requires **no local repository files** and imports nothing from local disks.

#### 🚀 Two-Stage Transfer Learning Strategy:
1. **Stage 1 (This Notebook)**: Pretrain the full **DGNet** architecture (MobileNetV3 backbone + Texture/Gradient Encoder + GIT + NCD Decoder) on the large-scale public **COD10K** camouflage benchmark (~10,000 images). This teaches the network general camouflage physics (boundary discontinuities, concealed contours, and terrain noise suppression).
2. **Stage 2 (Local Military Fine-Tuning)**: Export the resulting `dgnet_cod10k_pretrained.weights.h5` checkpoint back to your local repository (`models/raw_checkpoints/`) and fine-tune on your tactical military surveillance dataset (`dataset-splitM`).

#### 📚 Public Datasets (from dataset.txt):
* **Primary (COD10K)**: [https://www.kaggle.com/datasets/ivanomelchenkoim11/cod10k-dataset](https://www.kaggle.com/datasets/ivanomelchenkoim11/cod10k-dataset) *(Gold standard camouflage benchmark)*
* **Secondary (NC4K)**: [https://www.kaggle.com/datasets/ivanomelchenkoim11/nc4k-dataset](https://www.kaggle.com/datasets/ivanomelchenkoim11/nc4k-dataset)
* **Secondary (CAMO)**: [https://www.kaggle.com/datasets/danhnt/camo-fs-dataset](https://www.kaggle.com/datasets/danhnt/camo-fs-dataset)


## 1. Environment Setup & Hardware Configuration
Installs required packages (`kagglehub`, `opencv-python-headless`) and configures GPU memory growth.


In [ ]:
# 1. Install dependencies (kagglehub enables direct Kaggle dataset downloads in Colab)
!pip install -q kagglehub opencv-python-headless matplotlib

import os
import sys
import time
import shutil
import random
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union

import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, losses

print(f"TensorFlow Version: {tf.__version__}")

# Detect GPU Hardware Acceleration
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
            print(f"✓ GPU Detected: {gpu.name} (Memory growth enabled)")
        except Exception as e:
            print(f"GPU notice: {e}")
else:
    print("ℹ No physical GPU detected; running on CPU (To enable GPU in Colab: Runtime > Change runtime type > T4 GPU).")


## 2. Ingest Kaggle Camouflage Dataset (`COD10K`)
Downloads the benchmark dataset directly inside Google Colab using `kagglehub` (or Kaggle API).
Matches image files with ground truth segmentation masks automatically across subdirectories.


In [ ]:
# 2. Automated Kaggle Dataset Ingestion
# Selected benchmark from dataset.txt: ivanomelchenkoim11/cod10k-dataset
import kagglehub

print("[1/3] Downloading COD10K dataset from Kaggle via kagglehub...")
try:
    # kagglehub downloads public Kaggle datasets directly without needing manual kaggle.json
    kaggle_data_path = kagglehub.dataset_download("ivanomelchenkoim11/cod10k-dataset")
    print(f"✓ Downloaded to local path: {kaggle_data_path}")
except Exception as e_kh:
    print(f"Notice from kagglehub: {e_kh}")
    # Fallback to local /content path if uploaded manually or extracted elsewhere
    kaggle_data_path = "/content/cod10k-dataset"
    print(f"Using fallback directory: {kaggle_data_path}")

DATASET_ROOT = Path(kaggle_data_path)

# Recursive scanner to find and pair images with their ground truth masks
def discover_cod_pairs(root_dir: Path):
    valid_exts = {".jpg", ".jpeg", ".png", ".bmp"}
    all_files = list(root_dir.rglob("*"))
    
    img_candidates = {}
    gt_candidates = {}
    
    for f in all_files:
        if not f.is_file() or f.suffix.lower() not in valid_exts:
            continue
        p_str = str(f).replace("\\", "/")
        stem = f.stem
        
        # Ground truth masks typically reside in folders named 'GT', 'GT_Object', 'Mask', 'masks'
        is_gt = any(k in p_str for k in ["/GT_Object/", "/GT/", "/Masks/", "/masks/", "/gt/", "_GT_Object", "_GT"])
        
        if is_gt:
            gt_candidates[stem] = str(f)
        else:
            img_candidates[stem] = str(f)
            
    print(f"  • Found {len(img_candidates)} candidate images and {len(gt_candidates)} candidate masks.")
    
    matched_imgs, matched_masks = [], []
    for stem, ip in img_candidates.items():
        # Match exact stem or stem with GT suffix
        gp = gt_candidates.get(stem) or gt_candidates.get(f"{stem}_GT") or gt_candidates.get(f"{stem}_mask")
        if gp:
            matched_imgs.append(ip)
            matched_masks.append(gp)
            
    return matched_imgs, matched_masks

print("\n[2/3] Scanning dataset hierarchy for image-mask pairs...")
matched_imgs, matched_masks = discover_cod_pairs(DATASET_ROOT)
print(f"✓ Successfully matched {len(matched_imgs)} image-mask pairs!")

# Separate into Training and Validation/Testing splits (85% train, 15% val/test)
assert len(matched_imgs) > 0, f"No image-mask pairs matched in {DATASET_ROOT}. Please verify path."

combined = list(zip(matched_imgs, matched_masks))
random.seed(42)
random.shuffle(combined)

split_idx = int(0.85 * len(combined))
train_pairs = combined[:split_idx]
val_pairs = combined[split_idx:]

train_imgs, train_masks = [p[0] for p in train_pairs], [p[1] for p in train_pairs]
val_imgs, val_masks = [p[0] for p in val_pairs], [p[1] for p in val_pairs]

print(f"\n[3/3] Dataset Split Summary:")
print(f"  • Training Samples:   {len(train_imgs):,}")
print(f"  • Validation Samples: {len(val_imgs):,}")


## 3. Configuration & Hyperparameters
Define resolution, batch size, learning rate, and model paths.


In [ ]:
# 3. Hyperparameters & Configuration
IMG_SIZE = (384, 384)
BATCH_SIZE = 16          # Reduce to 8 if encountering GPU CUDA Out Of Memory on 16GB GPUs
LEARNING_RATE = 1e-4
EPOCHS = 30              # 30-40 epochs on COD10K provides excellent boundary representation
BACKBONE = "mobilenet_v3_large"  # Matches local DGNet architecture exactly (or 'efficientnet_b1')
GRADIENT_KERNEL = "sobel"
LAMBDA_GRAD_LOSS = 1.0

# Colab / Cloud Output directories
OUTPUT_DIR = Path("/content/dgnet_output")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

BEST_WEIGHTS_PATH = CHECKPOINT_DIR / "dgnet_cod10k_pretrained.weights.h5"
BEST_KERAS_PATH = CHECKPOINT_DIR / "dgnet_cod10k_pretrained_model.keras"

print(f"Image Resolution:    {IMG_SIZE}")
print(f"Batch Size:          {BATCH_SIZE}")
print(f"Learning Rate:       {LEARNING_RATE}")
print(f"Pretraining Epochs:  {EPOCHS}")
print(f"Backbone:            {BACKBONE}")
print(f"Output Checkpoints:  {CHECKPOINT_DIR}")


## 4. High-Throughput `tf.data` Pipeline & Augmentations
Applies Canny edge supervision, multi-scale Random Resized Crops, CutOut, geometric flips, and ImageNet standardization.


In [ ]:
# 4. Self-Contained High-Throughput Data Pipeline
IMAGENET_MEAN = tf.constant([0.485, 0.456, 0.406], dtype=tf.float32)
IMAGENET_STD = tf.constant([0.229, 0.224, 0.225], dtype=tf.float32)

def compute_object_gradient_np(image_uint8: np.ndarray, mask_uint8: np.ndarray) -> np.ndarray:
    # Official DGNet Canny object gradient supervision (cv2.Canny * mask)
    if image_uint8.ndim == 4:
        image_uint8 = image_uint8[0]
    if mask_uint8.ndim == 3 and mask_uint8.shape[-1] == 1:
        mask_uint8 = mask_uint8[:, :, 0]
    canny = cv2.Canny(image_uint8, 64, 128)
    fore = ((canny > 0) & (mask_uint8 >= 128)).astype(np.float32)
    return fore[:, :, np.newaxis]

def parse_sample(img_path: tf.Tensor, mask_path: tf.Tensor, image_size: tuple):
    img_bytes = tf.io.read_file(img_path)
    image_raw = tf.image.decode_image(img_bytes, channels=3, expand_animations=False)
    image = tf.image.resize(image_raw, image_size)
    image_uint8 = tf.cast(image, tf.uint8)

    mask_bytes = tf.io.read_file(mask_path)
    mask = tf.image.decode_image(mask_bytes, channels=1, expand_animations=False)
    mask = tf.image.resize(mask, image_size, method="nearest")
    mask_bin = tf.cast(mask >= 128, tf.float32)

    def _py_grad(im, m):
        return compute_object_gradient_np(im, m)

    grad_gt = tf.numpy_function(_py_grad, [image_uint8, tf.cast(mask, tf.uint8)], tf.float32)
    grad_gt.set_shape([image_size[0], image_size[1], 1])

    image_float = tf.cast(image, tf.float32) / 255.0
    image_float.set_shape([image_size[0], image_size[1], 3])
    mask_bin.set_shape([image_size[0], image_size[1], 1])

    return image_float, mask_bin, grad_gt

def _random_resized_crop(img, mask, grad_gt, target_size, scale_min=0.5, scale_max=1.0):
    h, w = target_size
    img_h, img_w = tf.shape(img)[0], tf.shape(img)[1]
    scale = tf.random.uniform((), minval=scale_min, maxval=scale_max)
    crop_h = tf.minimum(tf.cast(tf.cast(img_h, tf.float32) * tf.sqrt(scale), tf.int32), img_h)
    crop_w = tf.minimum(tf.cast(tf.cast(img_w, tf.float32) * tf.sqrt(scale), tf.int32), img_w)
    offset_h = tf.random.uniform((), minval=0, maxval=img_h - crop_h + 1, dtype=tf.int32)
    offset_w = tf.random.uniform((), minval=0, maxval=img_w - crop_w + 1, dtype=tf.int32)
    img = tf.image.crop_to_bounding_box(img, offset_h, offset_w, crop_h, crop_w)
    mask = tf.image.crop_to_bounding_box(mask, offset_h, offset_w, crop_h, crop_w)
    grad_gt = tf.image.crop_to_bounding_box(grad_gt, offset_h, offset_w, crop_h, crop_w)
    return tf.image.resize(img, [h, w]), tf.image.resize(mask, [h, w], method="nearest"), tf.image.resize(grad_gt, [h, w], method="nearest")

def _cutout_tf(img, max_fraction=0.25):
    h, w = tf.shape(img)[0], tf.shape(img)[1]
    cut_h = tf.cast(tf.cast(h, tf.float32) * tf.random.uniform((), 0.05, max_fraction), tf.int32)
    cut_w = tf.cast(tf.cast(w, tf.float32) * tf.random.uniform((), 0.05, max_fraction), tf.int32)
    cy, cx = tf.random.uniform((), minval=0, maxval=h, dtype=tf.int32), tf.random.uniform((), minval=0, maxval=w, dtype=tf.int32)
    y1, y2 = tf.maximum(cy - cut_h // 2, 0), tf.minimum(cy + cut_h // 2, h)
    x1, x2 = tf.maximum(cx - cut_w // 2, 0), tf.minimum(cx + cut_w // 2, w)
    rows, cols = tf.range(h), tf.range(w)
    keep_mask = tf.cast(tf.expand_dims(tf.logical_or(rows < y1, rows >= y2), 1) | tf.expand_dims(tf.logical_or(cols < x1, cols >= x2), 0), tf.float32)
    return img * tf.expand_dims(keep_mask, -1)

def create_tf_dataset(image_paths, mask_paths, image_size=(384, 384), batch_size=16, is_train=True):
    dataset = tf.data.Dataset.from_tensor_slices((image_paths, mask_paths))
    dataset = dataset.map(lambda x, y: parse_sample(x, y, image_size), num_parallel_calls=tf.data.AUTOTUNE)
    
    if is_train:
        dataset = dataset.shuffle(buffer_size=min(len(image_paths), 512), seed=42)
        
        def _augment_fn(img, mask, grad_gt):
            if tf.random.uniform(()) > 0.5:
                img, mask, grad_gt = _random_resized_crop(img, mask, grad_gt, image_size)
            if tf.random.uniform(()) > 0.5:
                img = tf.image.flip_left_right(img)
                mask = tf.image.flip_left_right(mask)
                grad_gt = tf.image.flip_left_right(grad_gt)
            if tf.random.uniform(()) > 0.5:
                img = tf.image.flip_up_down(img)
                mask = tf.image.flip_up_down(mask)
                grad_gt = tf.image.flip_up_down(grad_gt)
            k = tf.random.uniform((), minval=0, maxval=4, dtype=tf.int32)
            img = tf.image.rot90(img, k=k)
            mask = tf.image.rot90(mask, k=k)
            grad_gt = tf.image.rot90(grad_gt, k=k)
            if tf.random.uniform(()) > 0.7:
                img = _cutout_tf(img)
            img = tf.image.random_brightness(img, max_delta=0.2)
            img = tf.image.random_contrast(img, lower=0.8, upper=1.2)
            img = tf.clip_by_value(img, 0.0, 1.0)
            img_norm = (img - IMAGENET_MEAN) / IMAGENET_STD
            return img_norm, mask, grad_gt
            
        dataset = dataset.map(_augment_fn, num_parallel_calls=tf.data.AUTOTUNE)
    else:
        dataset = dataset.map(lambda img, m, g: ((img - IMAGENET_MEAN) / IMAGENET_STD, m, g), num_parallel_calls=tf.data.AUTOTUNE)
        
    dataset = dataset.batch(batch_size, drop_remainder=False)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

print("✓ Data pipeline configured successfully.")


## 5. Full DGNet Architecture (Pure TensorFlow 2.x)
Directly defined in this notebook with **100% identical layer hierarchy and naming** as `src/models/dgnet.py`.
This guarantees that weights saved here will load seamlessly into your local repository without any missing keys or weight shape mismatches.


In [ ]:
# 5. DGNet Model Architecture (Pure TensorFlow 2.x)

class ConvBR(layers.Layer):
    # Standard Convolution + Batch Normalization + ReLU block with He Normal initialization
    def __init__(self, out_channel: int, kernel_size: int = 3, stride: int = 1, padding: str = "same", dilation_rate: int = 1, name: Optional[str] = None, **kwargs):
        super().__init__(name=name, **kwargs)
        self.conv = layers.Conv2D(
            filters=out_channel, kernel_size=kernel_size, strides=stride, padding=padding,
            dilation_rate=dilation_rate, use_bias=False, kernel_initializer="he_normal",
            name=f"{name}_conv" if name else None
        )
        self.bn = layers.BatchNormalization(name=f"{name}_bn" if name else None)
        self.relu = layers.ReLU(name=f"{name}_relu" if name else None)

    def call(self, x: tf.Tensor, training: Optional[bool] = None) -> tf.Tensor:
        return self.relu(self.bn(self.conv(x), training=training))


class CBAM(layers.Layer):
    # Convolutional Block Attention Module: Channel Attention + Spatial Attention
    def __init__(self, channels: int, reduction: int = 16, name: Optional[str] = None, **kwargs):
        super().__init__(name=name, **kwargs)
        reduced_ch = max(channels // reduction, 8)
        self.fc1 = layers.Dense(reduced_ch, activation="relu", use_bias=False, name=f"{name}_fc1" if name else None)
        self.fc2 = layers.Dense(channels, use_bias=False, name=f"{name}_fc2" if name else None)
        self.spatial_conv = layers.Conv2D(1, kernel_size=7, padding="same", activation="sigmoid", use_bias=False, name=f"{name}_spconv" if name else None)

    def call(self, x: tf.Tensor) -> tf.Tensor:
        avg_p = tf.reduce_mean(x, axis=[1, 2], keepdims=True)
        max_p = tf.reduce_max(x, axis=[1, 2], keepdims=True)
        ch_attn = tf.nn.sigmoid(self.fc2(self.fc1(avg_p)) + self.fc2(self.fc1(max_p)))
        x = x * ch_attn
        avg_s = tf.reduce_mean(x, axis=-1, keepdims=True)
        max_s = tf.reduce_max(x, axis=-1, keepdims=True)
        sp_attn = self.spatial_conv(tf.concat([avg_s, max_s], axis=-1))
        return x * sp_attn


class DimensionalReduction(layers.Layer):
    # Dimensional reduction per scale with CBAM attention and SpatialDropout
    def __init__(self, out_channel: int, use_cbam: bool = True, dropout_rate: float = 0.1, name: Optional[str] = None, **kwargs):
        super().__init__(name=name, **kwargs)
        self.reduce1 = ConvBR(out_channel, kernel_size=3, padding="same", name=f"{name}_br1" if name else None)
        self.reduce2 = ConvBR(out_channel, kernel_size=3, padding="same", name=f"{name}_br2" if name else None)
        self.cbam = CBAM(out_channel, name=f"{name}_cbam" if name else None) if use_cbam else None
        self.spatial_dropout = layers.SpatialDropout2D(rate=dropout_rate, name=f"{name}_sdrop" if name else None) if dropout_rate > 0 else None

    def call(self, x: tf.Tensor, training: Optional[bool] = None) -> tf.Tensor:
        x = self.reduce1(x, training=training)
        x = self.reduce2(x, training=training)
        if self.cbam is not None:
            x = self.cbam(x)
        if self.spatial_dropout is not None:
            x = self.spatial_dropout(x, training=training)
        return x


class SoftGroupingStrategy(layers.Layer):
    # Soft Grouping Strategy (SGS): 3 parallel 1x1 grouped convolutions (N=[4, 8, 16])
    def __init__(self, in_channel: int, out_channel: int, N: List[int] = [4, 8, 16], name: Optional[str] = None, **kwargs):
        super().__init__(name=name, **kwargs)
        self.g_conv1 = layers.Conv2D(out_channel, kernel_size=1, groups=N[0], use_bias=False, kernel_initializer="he_normal")
        self.g_conv2 = layers.Conv2D(out_channel, kernel_size=1, groups=N[1], use_bias=False, kernel_initializer="he_normal")
        self.g_conv3 = layers.Conv2D(out_channel, kernel_size=1, groups=N[2], use_bias=False, kernel_initializer="he_normal")

    def call(self, q: tf.Tensor) -> tf.Tensor:
        return self.g_conv1(q) + self.g_conv2(q) + self.g_conv3(q)


class GradientInducedTransition(layers.Layer):
    # Gradient-Induced Transition (GIT) with Soft Grouping Strategy
    def __init__(self, channel: int = 64, M: List[int] = [8, 8, 8], N: List[int] = [4, 8, 16], name: Optional[str] = None, **kwargs):
        super().__init__(name=name, **kwargs)
        self.M = M
        self.sgs3 = SoftGroupingStrategy(in_channel=channel + 32, out_channel=channel, N=N, name=f"{name}_sgs3" if name else None)
        self.sgs4 = SoftGroupingStrategy(in_channel=channel + 32, out_channel=channel, N=N, name=f"{name}_sgs4" if name else None)
        self.sgs5 = SoftGroupingStrategy(in_channel=channel + 32, out_channel=channel, N=N, name=f"{name}_sgs5" if name else None)

    def _interleave(self, xr: tf.Tensor, xg: tf.Tensor, M: int) -> tf.Tensor:
        xr_g = tf.split(xr, num_or_size_splits=M, axis=-1)
        xg_g = tf.split(xg, num_or_size_splits=M, axis=-1)
        interleaved = []
        for i in range(M):
            interleaved.extend([xr_g[i], xg_g[i]])
        return tf.concat(interleaved, axis=-1)

    def call(self, xr3: tf.Tensor, xr4: tf.Tensor, xr5: tf.Tensor, xg: tf.Tensor, training: Optional[bool] = None):
        h4, w4 = tf.shape(xr4)[1], tf.shape(xr4)[2]
        h5, w5 = tf.shape(xr5)[1], tf.shape(xr5)[2]
        xg4 = tf.image.resize(xg, (h4, w4), method="bilinear")
        xg5 = tf.image.resize(xg, (h5, w5), method="bilinear")
        zt3 = xr3 + self.sgs3(self._interleave(xr3, xg, self.M[0]))
        zt4 = xr4 + self.sgs4(self._interleave(xr4, xg4, self.M[1]))
        zt5 = xr5 + self.sgs5(self._interleave(xr5, xg5, self.M[2]))
        return zt3, zt4, zt5


class NeighborConnectionDecoder(layers.Layer):
    # Neighbor Connection Decoder (NCD) with Deep Supervision
    def __init__(self, channel: int = 64, deep_supervision: bool = True, dropout_rate: float = 0.15, name: Optional[str] = None, **kwargs):
        super().__init__(name=name, **kwargs)
        self.deep_supervision = deep_supervision
        self.conv_upsample1 = ConvBR(channel, kernel_size=3, padding="same", name=f"{name}_u1" if name else None)
        self.conv_upsample2 = ConvBR(channel, kernel_size=3, padding="same", name=f"{name}_u2" if name else None)
        self.conv_upsample3 = ConvBR(channel, kernel_size=3, padding="same", name=f"{name}_u3" if name else None)
        self.conv_upsample4 = ConvBR(channel, kernel_size=3, padding="same", name=f"{name}_u4" if name else None)
        self.conv_upsample5 = ConvBR(channel, kernel_size=3, padding="same", name=f"{name}_u5" if name else None)
        self.conv_concat2 = ConvBR(channel, kernel_size=3, padding="same", name=f"{name}_c2" if name else None)
        self.conv_concat3 = ConvBR(channel, kernel_size=3, padding="same", name=f"{name}_c3" if name else None)
        self.conv4 = ConvBR(channel, kernel_size=3, padding="same", name=f"{name}_c4" if name else None)
        self.decoder_dropout = layers.Dropout(rate=dropout_rate, name=f"{name}_drop" if name else None) if dropout_rate > 0 else None
        self.conv5 = layers.Conv2D(1, kernel_size=1, use_bias=True, kernel_initializer="he_normal", name=f"{name}_c5" if name else None)
        if self.deep_supervision:
            self.aux_conv4 = layers.Conv2D(1, kernel_size=1, use_bias=True, kernel_initializer="he_normal", name=f"{name}_aux4" if name else None)
            self.aux_conv5 = layers.Conv2D(1, kernel_size=1, use_bias=True, kernel_initializer="he_normal", name=f"{name}_aux5" if name else None)

    def _resize(self, x: tf.Tensor, target: tf.Tensor) -> tf.Tensor:
        return tf.image.resize(x, (tf.shape(target)[1], tf.shape(target)[2]), method="bilinear")

    def call(self, zt5: tf.Tensor, zt4: tf.Tensor, zt3: tf.Tensor, training: Optional[bool] = None):
        zt5_1 = zt5
        zt4_1 = self.conv_upsample1(self._resize(zt5, zt4), training=training) * zt4
        zt3_1 = self.conv_upsample2(self._resize(zt4_1, zt3), training=training) * self.conv_upsample3(self._resize(zt4, zt3), training=training) * zt3
        zt4_2 = self.conv_concat2(tf.concat([zt4_1, self.conv_upsample4(self._resize(zt5_1, zt4), training=training)], axis=-1), training=training)
        zt3_2 = self.conv_concat3(tf.concat([zt3_1, self.conv_upsample5(self._resize(zt4_2, zt3), training=training)], axis=-1), training=training)
        pc = self.conv4(zt3_2, training=training)
        if self.decoder_dropout is not None:
            pc = self.decoder_dropout(pc, training=training)
        pc = self.conv5(pc)
        if self.deep_supervision:
            return pc, self.aux_conv4(zt4_2), self.aux_conv5(zt5_1)
        return pc


class TextureEncoder(layers.Layer):
    # Processes raw RGB into gradient feature xg (stride 8, 32 ch) and gradient map pg
    def __init__(self, name: Optional[str] = None, **kwargs):
        super().__init__(name=name, **kwargs)
        self.conv1 = ConvBR(64, kernel_size=7, stride=2, padding="same", name=f"{name}_c1" if name else None)
        self.conv2 = ConvBR(64, kernel_size=3, stride=2, padding="same", name=f"{name}_c2" if name else None)
        self.conv3 = ConvBR(32, kernel_size=3, stride=2, padding="same", name=f"{name}_c3" if name else None)
        self.conv_out = layers.Conv2D(1, kernel_size=1, strides=1, padding="same", use_bias=True, kernel_initializer="he_normal", name=f"{name}_out" if name else None)

    def call(self, x: tf.Tensor, training: Optional[bool] = None):
        feat = self.conv3(self.conv2(self.conv1(x, training=training), training=training), training=training)
        return feat, self.conv_out(feat)


class ContextEncoder(layers.Layer):
    # Multi-scale context encoder with ImageNet pretrained backbone at strides 8, 16, 32
    def __init__(self, backbone_name: str = "mobilenet_v3_large", input_shape: Tuple[int, int, int] = (384, 384, 3), pretrained: bool = True, name: Optional[str] = None, **kwargs):
        super().__init__(name=name, **kwargs)
        self.backbone_name = backbone_name.lower().replace("-", "_")
        weights = "imagenet" if pretrained else None
        
        if "resnet50" in self.backbone_name:
            base = tf.keras.applications.ResNet50(include_top=False, weights=weights, input_shape=input_shape)
            x3_name, x4_name, x5_name = "conv3_block4_out", "conv4_block6_out", "conv5_block3_out"
        elif "efficientnet_b1" in self.backbone_name:
            base = tf.keras.applications.EfficientNetB1(include_top=False, weights=weights, input_shape=input_shape)
            x3_name, x4_name, x5_name = "block3c_add", "block5d_add", "block7b_add"
        else: # MobileNetV3Large default
            base = tf.keras.applications.MobileNetV3Large(include_top=False, weights=weights, input_shape=input_shape)
            layer_map = {l.name: l for l in base.layers}
            x3_candidates = ["expanded_conv_5_add", "expanded_conv_3_project_bn", "expanded_conv_3/Add", "expanded_conv_3_add"]
            x4_candidates = ["expanded_conv_11_add", "expanded_conv_11/Add", "expanded_conv_11_project_bn"]
            x5_candidates = ["expanded_conv_14_add", "expanded_conv_14_project_bn", "expanded_conv_14/Add", "conv_1"]
            x3_name = next(c for c in x3_candidates if c in layer_map)
            x4_name = next(c for c in x4_candidates if c in layer_map)
            x5_name = next(c for c in x5_candidates if c in layer_map)

        self.extractor = models.Model(
            inputs=base.input,
            outputs=[base.get_layer(x3_name).output, base.get_layer(x4_name).output, base.get_layer(x5_name).output],
            name=f"{self.backbone_name}_multi_scale"
        )

    def call(self, x: tf.Tensor, training: Optional[bool] = None):
        return self.extractor(x, training=training)


class DGNet(models.Model):
    # Complete DGNet Architecture matching official paper (MIR 2023)
    def __init__(self, backbone_name: str = "mobilenet_v3_large", input_shape: Tuple[int, int, int] = (384, 384, 3), channel: int = 64, M: List[int] = [8, 8, 8], N: List[int] = [4, 8, 16], pretrained: bool = True, **kwargs):
        super().__init__(**kwargs)
        self.context_encoder = ContextEncoder(backbone_name=backbone_name, input_shape=input_shape, pretrained=pretrained, name="context_encoder")
        self.texture_encoder = TextureEncoder(name="texture_encoder")
        self.dr3 = DimensionalReduction(out_channel=channel, name="dr3")
        self.dr4 = DimensionalReduction(out_channel=channel, name="dr4")
        self.dr5 = DimensionalReduction(out_channel=channel, name="dr5")
        self.git = GradientInducedTransition(channel=channel, M=M, N=N, name="git")
        self.ncd = NeighborConnectionDecoder(channel=channel, name="ncd")

    def call(self, inputs: tf.Tensor, training: Optional[bool] = None, return_dict: bool = False):
        target_h, target_w = tf.shape(inputs)[1], tf.shape(inputs)[2]
        x3, x4, x5 = self.context_encoder(inputs, training=training)
        xr3 = self.dr3(x3, training=training)
        xr4 = self.dr4(x4, training=training)
        xr5 = self.dr5(x5, training=training)
        xg, pg = self.texture_encoder(inputs, training=training)
        zt3, zt4, zt5 = self.git(xr3, xr4, xr5, xg, training=training)
        ncd_out = self.ncd(zt5, zt4, zt3, training=training)
        pc, pc4, pc5 = ncd_out if isinstance(ncd_out, tuple) else (ncd_out, None, None)
        
        mask_logits = tf.image.resize(pc, (target_h, target_w), method="bilinear")
        grad_logits = tf.image.resize(pg, (target_h, target_w), method="bilinear")

        if training or return_dict:
            res = {"mask_pred": mask_logits, "grad_pred": grad_logits}
            if pc4 is not None: res["mask_pred_aux4"] = tf.image.resize(pc4, (target_h, target_w), method="bilinear")
            if pc5 is not None: res["mask_pred_aux5"] = tf.image.resize(pc5, (target_h, target_w), method="bilinear")
            return res
        return tf.nn.sigmoid(mask_logits)

def build_dgnet_model(backbone_name="mobilenet_v3_large", input_shape=(384, 384, 3), channel=64, pretrained=True, **kwargs):
    return DGNet(backbone_name=backbone_name, input_shape=input_shape, channel=channel, pretrained=pretrained, **kwargs)

print("✓ DGNet architecture compiled.")


## 6. Multi-Task Structure Loss & Optimizer
Implements Weighted BCE + Weighted IoU with 31x31 adaptive edge weighting + Boundary IoU loss + gradient MSE.


In [ ]:
# 6. DGNet Multi-Task Structure Loss
class DGNetLoss(losses.Loss):
    def __init__(self, lambda_grad_loss: float = 1.0, label_smoothing: float = 0.02, name: str = "dgnet_loss", **kwargs):
        super().__init__(name=name)
        self.lambda_grad_loss = float(lambda_grad_loss)
        self.label_smoothing = float(label_smoothing)
        self.laplacian_kernel = tf.reshape(tf.constant([[-1, -1, -1], [-1, 8, -1], [-1, -1, -1]], dtype=tf.float32), [3, 3, 1, 1])

    def structure_loss(self, y_true: tf.Tensor, y_pred_logits: tf.Tensor) -> tf.Tensor:
        y_true_f = tf.cast(y_true, tf.float32)
        if self.label_smoothing > 0:
            y_true_f = y_true_f * (1.0 - self.label_smoothing) + self.label_smoothing / 2.0
        avg_pool = tf.nn.avg_pool2d(y_true_f, ksize=[1, 31, 31, 1], strides=[1, 1, 1, 1], padding="SAME")
        weit = 1.0 + 5.0 * tf.abs(avg_pool - y_true_f)

        bce_raw = tf.nn.sigmoid_cross_entropy_with_logits(labels=y_true_f, logits=y_pred_logits)
        wbce = tf.reduce_sum(weit * bce_raw, axis=[1, 2, 3]) / (tf.reduce_sum(weit, axis=[1, 2, 3]) + 1e-8)

        pred_sig = tf.nn.sigmoid(y_pred_logits)
        inter = tf.reduce_sum((pred_sig * y_true_f) * weit, axis=[1, 2, 3])
        union = tf.reduce_sum((pred_sig + y_true_f) * weit, axis=[1, 2, 3])
        wiou = 1.0 - (inter + 1.0) / (union - inter + 1.0)
        return tf.reduce_mean(wbce + wiou)

    def boundary_loss(self, y_true: tf.Tensor, y_pred_logits: tf.Tensor) -> tf.Tensor:
        y_true_f = tf.cast(y_true, tf.float32)
        pred_sig = tf.nn.sigmoid(y_pred_logits)
        gt_edges = tf.clip_by_value(tf.abs(tf.nn.conv2d(y_true_f, self.laplacian_kernel, strides=[1, 1, 1, 1], padding="SAME")), 0.0, 1.0)
        pred_edges = tf.clip_by_value(tf.abs(tf.nn.conv2d(pred_sig, self.laplacian_kernel, strides=[1, 1, 1, 1], padding="SAME")), 0.0, 1.0)
        inter = tf.reduce_sum(gt_edges * pred_edges, axis=[1, 2, 3])
        union = tf.reduce_sum(gt_edges + pred_edges - gt_edges * pred_edges, axis=[1, 2, 3])
        return tf.reduce_mean(1.0 - (inter + 1.0) / (union + 1.0))

    def call(self, y_true: tf.Tensor, y_pred_dict: Dict[str, tf.Tensor]) -> tf.Tensor:
        mask_logits = y_pred_dict["mask_pred"]
        grad_logits = y_pred_dict["grad_pred"]
        seg_loss = self.structure_loss(y_true, mask_logits)
        if "mask_pred_aux4" in y_pred_dict: seg_loss += 0.5 * self.structure_loss(y_true, y_pred_dict["mask_pred_aux4"])
        if "mask_pred_aux5" in y_pred_dict: seg_loss += 0.25 * self.structure_loss(y_true, y_pred_dict["mask_pred_aux5"])
        seg_loss += 0.5 * self.boundary_loss(y_true, mask_logits)
        gt_grad = y_pred_dict.get("gt_grad")
        if gt_grad is not None:
            grad_loss = tf.reduce_mean(tf.square(tf.nn.sigmoid(grad_logits) - tf.cast(gt_grad, tf.float32)))
            return seg_loss + self.lambda_grad_loss * grad_loss
        return seg_loss

# Learning Rate Warmup + Cosine Decay
class WarmupCosineDecay(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, initial_lr, warmup_steps, total_steps):
        super().__init__()
        self.initial_lr = initial_lr
        self.warmup_steps = float(warmup_steps)
        self.cosine_decay = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=initial_lr, decay_steps=int(total_steps - warmup_steps))
    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warmup_lr = self.initial_lr * (step / tf.maximum(self.warmup_steps, 1.0))
        cosine_lr = self.cosine_decay(step - self.warmup_steps)
        return tf.where(step < self.warmup_steps, warmup_lr, cosine_lr)

print("✓ Loss functions and optimization schedules compiled.")


## 7. Model Instantiation & Benchmark Pretraining
Instantiates DGNet with ImageNet weights and executes pretraining on the Kaggle COD10K dataset.
Checkpoints are saved whenever validation loss improves.


In [ ]:
# 7. Model Instantiation & Pretraining Loop
model = build_dgnet_model(
    backbone_name=BACKBONE,
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3),
    channel=64,
    pretrained=True
)

# Run dummy input through model to initialize weight shapes
_ = model(tf.zeros((1, IMG_SIZE[0], IMG_SIZE[1], 3), dtype=tf.float32), training=False)
total_params = model.count_params()
print(f"Total Parameters: {total_params:,} ({total_params*4 / (1024*1024):.2f} MB)")

loss_fn = DGNetLoss(lambda_grad_loss=LAMBDA_GRAD_LOSS, label_smoothing=0.02)
total_steps = EPOCHS * max(len(train_imgs) // BATCH_SIZE, 1)
warmup_steps = min(5 * max(len(train_imgs) // BATCH_SIZE, 1), total_steps // 5)
lr_schedule = WarmupCosineDecay(LEARNING_RATE, warmup_steps, total_steps)
optimizer = tf.keras.optimizers.AdamW(learning_rate=lr_schedule, weight_decay=1e-4, clipnorm=1.0)

@tf.function
def train_step(x_b, y_m, y_g):
    with tf.GradientTape() as tape:
        preds = model(x_b, training=True, return_dict=True)
        preds["gt_grad"] = y_g
        loss = loss_fn(y_m, preds)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

@tf.function
def val_step(x_b, y_m, y_g):
    val_preds = model(x_b, training=False, return_dict=True)
    val_preds["gt_grad"] = y_g
    v_loss = loss_fn(y_m, val_preds)
    prob_mask = tf.nn.sigmoid(val_preds["mask_pred"])
    return v_loss, prob_mask

# Set to True to start pretraining
RUN_PRETRAINING = True

if RUN_PRETRAINING:
    print(f"\n{'='*70}\n Starting DGNet Pretraining on COD10K ({len(train_imgs):,} training images)\n{'='*70}")
    train_dataset = create_tf_dataset(train_imgs, train_masks, image_size=IMG_SIZE, batch_size=BATCH_SIZE, is_train=True)
    val_dataset = create_tf_dataset(val_imgs, val_masks, image_size=IMG_SIZE, batch_size=BATCH_SIZE, is_train=False)
    
    total_batches = len(train_imgs) // BATCH_SIZE
    best_val_loss = float("inf")
    history = {"train_loss": [], "val_loss": []}
    
    for epoch in range(1, EPOCHS + 1):
        epoch_loss = 0.0
        batches = 0
        t0 = time.time()
        
        for step, (x_b, y_m, y_g) in enumerate(train_dataset):
            loss = train_step(x_b, y_m, y_g)
            loss_val = float(loss)
            epoch_loss += loss_val
            batches += 1
            if batches % 50 == 0 or batches == total_batches:
                print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] Batch [{batches:03d}/{total_batches:03d}] | Step Loss: {loss_val:.4f} | Running Avg: {epoch_loss/batches:.4f}")
                
        avg_train_loss = epoch_loss / max(batches, 1)
        
        # Validation Pass
        val_loss_total = 0.0
        val_batches = 0
        for vx_b, vy_m, vy_g in val_dataset:
            v_loss, _ = val_step(vx_b, vy_m, vy_g)
            val_loss_total += float(v_loss)
            val_batches += 1
        avg_val_loss = val_loss_total / max(val_batches, 1)
        
        elapsed = time.time() - t0
        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(avg_val_loss)
        
        print(f"--> Epoch {epoch:02d} Complete | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Time: {elapsed:.1f}s")
        
        # Save Checkpoint whenever validation loss improves
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            model.save_weights(str(BEST_WEIGHTS_PATH))
            print(f"  ★ New Best Weights Saved: {BEST_WEIGHTS_PATH.name} (Val Loss: {best_val_loss:.4f})")
            
    print(f"\n✓ Pretraining finished! Best Validation Loss: {best_val_loss:.4f}")
else:
    print("ℹ Pretraining skipped (set RUN_PRETRAINING = True to execute).")


## 8. Quantitative Evaluation on Validation/Test Set
Calculates official Camouflaged Object Detection metrics: Pixel Accuracy, MAE, IoU, Dice (F1), and $F_\beta$ ($F_{0.3}$).


In [ ]:
# 8. Quantitative COD Benchmark Metrics Evaluation
class CODMetricsAccumulator:
    def __init__(self, threshold: float = 0.5, beta_sq: float = 0.3):
        self.threshold = threshold
        self.beta_sq = beta_sq
        self.tp = 0.0
        self.tn = 0.0
        self.fp = 0.0
        self.fn = 0.0
        self.mae_sum = 0.0
        self.pixel_count = 0
        self.sample_count = 0

    def update(self, y_true: tf.Tensor, y_pred_prob: tf.Tensor):
        y_true_f = tf.cast(y_true, tf.float32)
        y_pred_bin = tf.cast(y_pred_prob >= self.threshold, tf.float32)
        self.tp += float(tf.reduce_sum(y_pred_bin * y_true_f))
        self.tn += float(tf.reduce_sum((1.0 - y_pred_bin) * (1.0 - y_true_f)))
        self.fp += float(tf.reduce_sum(y_pred_bin * (1.0 - y_true_f)))
        self.fn += float(tf.reduce_sum((1.0 - y_pred_bin) * y_true_f))
        self.mae_sum += float(tf.reduce_sum(tf.abs(y_pred_prob - y_true_f)))
        self.pixel_count += int(tf.size(y_true_f))
        self.sample_count += int(y_true_f.shape[0])

    def result(self) -> Dict[str, float]:
        tot = self.tp + self.tn + self.fp + self.fn
        acc = (self.tp + self.tn) / tot if tot > 0 else 0.0
        mae = self.mae_sum / self.pixel_count if self.pixel_count > 0 else 0.0
        prec = self.tp / (self.tp + self.fp) if (self.tp + self.fp) > 0 else 0.0
        rec = self.tp / (self.tp + self.fn) if (self.tp + self.fn) > 0 else 0.0
        spec = self.tn / (self.tn + self.fp) if (self.tn + self.fp) > 0 else 0.0
        iou = self.tp / (self.tp + self.fp + self.fn) if (self.tp + self.fp + self.fn) > 0 else 0.0
        dice = 2.0 * self.tp / (2.0 * self.tp + self.fp + self.fn) if (2.0 * self.tp + self.fp + self.fn) > 0 else 0.0
        f_beta_denom = self.beta_sq * prec + rec
        f_beta = (1.0 + self.beta_sq) * prec * rec / f_beta_denom if f_beta_denom > 0 else 0.0
        return {
            "pixel_accuracy": acc, "mae": mae, "iou": iou, "dice": dice,
            "f_measure": f_beta, "precision": prec, "recall": rec, "specificity": spec,
            "sample_count": self.sample_count
        }

if BEST_WEIGHTS_PATH.exists():
    model.load_weights(str(BEST_WEIGHTS_PATH))
    print(f"✓ Loaded best weights for benchmark evaluation: {BEST_WEIGHTS_PATH.name}")

eval_dataset = create_tf_dataset(val_imgs[:500], val_masks[:500], image_size=IMG_SIZE, batch_size=8, is_train=False)
accumulator = CODMetricsAccumulator(threshold=0.5, beta_sq=0.3)
t_start = time.perf_counter()

for x_b, y_m, _ in eval_dataset:
    preds = model(x_b, training=False)
    accumulator.update(y_m, preds)

total_eval_time = time.perf_counter() - t_start
metrics = accumulator.result()
fps = metrics["sample_count"] / total_eval_time if total_eval_time > 0 else 0.0

print("\n" + "=" * 65)
print("       DGNet Pretrained Model COD Benchmark Evaluation")
print("=" * 65)
print(f" Evaluated Samples:       {metrics['sample_count']}")
print(f" Pixel Accuracy:          {metrics['pixel_accuracy']*100:.2f}%")
print(f" Mean Absolute Error:     {metrics['mae']:.4f}")
print(f" IoU (Jaccard Index):     {metrics['iou']:.4f}")
print(f" Dice Coefficient (F1):   {metrics['dice']:.4f}")
print(f" F-measure (Fβ=0.3):     {metrics['f_measure']:.4f}")
print(f" Precision:               {metrics['precision']:.4f}")
print(f" Recall:                  {metrics['recall']:.4f}")
print(f" Specificity:             {metrics['specificity']:.4f}")
print(f" Inference Speed:         {fps:.1f} FPS")
print("=" * 65)


## 9. Visual Inspection: Test Sample Predictions
Visualizes raw input images, ground truth camouflage masks, and model predictions side-by-side.


In [ ]:
# 9. Visual Verification on Samples
num_samples = min(4, len(val_imgs))
fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4 * num_samples))

for i in range(num_samples):
    img_raw = cv2.cvtColor(cv2.imread(val_imgs[i]), cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_raw, IMG_SIZE)
    img_norm = (img_resized.astype(np.float32) / 255.0 - IMAGENET_MEAN.numpy()) / IMAGENET_STD.numpy()
    
    gt_raw = cv2.imread(val_masks[i], cv2.IMREAD_GRAYSCALE)
    gt_bin = (cv2.resize(gt_raw, IMG_SIZE, interpolation=cv2.INTER_NEAREST) >= 128).astype(np.uint8) * 255
    
    inp_t = tf.expand_dims(tf.convert_to_tensor(img_norm, dtype=tf.float32), 0)
    pred_prob = model(inp_t, training=False).numpy()[0, :, :, 0]
    pred_bin = (pred_prob >= 0.5).astype(np.uint8) * 255
    
    row_axes = axes[i] if num_samples > 1 else axes
    row_axes[0].imshow(img_resized)
    row_axes[0].set_title(f"Sample {i+1}: Input Image")
    row_axes[0].axis("off")
    
    row_axes[1].imshow(gt_bin, cmap="gray")
    row_axes[1].set_title("Ground Truth Mask")
    row_axes[1].axis("off")
    
    row_axes[2].imshow(pred_bin, cmap="gray")
    row_axes[2].set_title("DGNet Pretrained Prediction")
    row_axes[2].axis("off")

plt.tight_layout()
plt.show()


## 10. Download Single Best Checkpoint for Retraining
Download only the single best `.weights.h5` checkpoint file directly to your local computer with one click.


In [ ]:
# 10. Download Single Best Checkpoint to Local Computer
# Trigger browser download if running in Google Colab
try:
    from google.colab import files
    print(f"Downloading single best checkpoint: {BEST_WEIGHTS_PATH.name}...")
    files.download(str(BEST_WEIGHTS_PATH))
except ImportError:
    print(f"ℹ Not in Google Colab. Download manually from: {BEST_WEIGHTS_PATH.resolve()}")


## 11. How to Retrain / Fine-Tune on Your Military Dataset (Stage 2)

Once you download `dgnet_cod10k_pretrained.weights.h5`, follow these 3 simple steps on your local machine:

### Step 1: Copy weights to local repo
Copy the downloaded file into your local project directory:
```powershell
Copy-Item "$HOME\Downloads\dgnet_cod10k_pretrained.weights.h5" -Destination "c:\Users\Acer\Desktop\COD_PROJECT\models\raw_checkpoints\"
```

### Step 2: Fine-Tune using `scripts/DGnet.py`
In your local terminal or PowerShell, run:
```powershell
python scripts/DGnet.py --resume models/raw_checkpoints/dgnet_cod10k_pretrained.weights.h5 --epochs 30 --lr 0.00002
```
* Or in Python code / notebook:
```python
# Instantiate model
model = build_dgnet_model(backbone_name="mobilenet_v3_large", pretrained=True)
_ = model(tf.zeros((1, 384, 384, 3)))

# Load COD10K pretrained weights
model.load_weights("models/raw_checkpoints/dgnet_cod10k_pretrained.weights.h5")
print("✓ Loaded COD10K pretrained weights! Ready for military fine-tuning.")

# Train on data/raw/dataset-splitM with lower learning rate (e.g. 2e-5)
```

Because the network already mastered general camouflage physics on COD10K, fine-tuning on your ~748 tactical images will converge rapidly without overfitting or background noise speckles!
